In [1]:
import gym
import highway_env
import time
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from highway_env.tb_callback import TensorboardCallback
from tqdm.rich import tqdm
from stable_baselines3.common.env_checker import check_env


OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output/'
N_AGENTS = 2
total_timesteps = int(2e4)

n_cpu = 6
batch_size = 64
env = gym.make("dm-env-v0")
check_env(env)
env.configure({
    "controlled_vehicles": N_AGENTS,
    "vehicles_count": 10,
    "training_total_timesteps": total_timesteps
})

adv_model = PPO("MlpPolicy",
        env,
        policy_kwargs=dict(net_arch=[dict(pi=[256, 256], vf=[256, 256])]),
        seed=123,
        gamma=0.8,
        batch_size=64,
        n_epochs=10,
        ent_coef=0.1,
        verbose=0,
        learning_rate=5e-4,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

victim_model = PPO.load('/home/pigo/HighwayDRL/final_models/ppo_standard_200k_FORNO.zip')

checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                                name_prefix='adv_highway_ppo')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                                    log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                                    deterministic=True, render=False)

tb_callback = TensorboardCallback()

try:
    for _ in tqdm(range(total_timesteps)):
        obs = env.reset()
        done = False
        while not done:
            
            victim_action, _ = _model.predict(obs, deterministic=True)
            
            vobs, vreward, done, _ = env.step(int(victim_action))
            # Train the adversarial agent
            adv_model.learn(total_timesteps=1)
            env.render()

finally:
    # Save the adversarial agent
    adv_model.save(f'{OUTPUT_PATH}/models/highway_ppo_adv')
    env.close()

/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/policies.py:458: UserWarning: As shared layers in the mlp_extractor are deprecated and will be removed in SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


Output()

KeyboardInterrupt: 

In [ ]:
# for episode in range(50):
#     obs = env.reset()
#     done = truncated = False
#     while not (done or truncated):
#         # Dispatch the observations to the model to get the tuple of actions
#         action = tuple(model.predict(obs_i) for obs_i in obs)
#         # Execute the actions
#         next_obs, reward, done, truncated, info = env.step(action)
#         # Update the model with the transitions observed by each agent
#         for obs_i, action_i, next_obs_i in zip(obs, action, next_obs):
#           model.update(obs_i, action_i, next_obs_i, reward, info, done, truncated)
#         obs = next_obs

# env.close()


# env.reset()
# done = False
# while not done:
#     observation, reward, done, info = env.step(env.action_space.sample())
#     env.render()
    
    # if env.controlled_vehicles[0].lane_index[2]>0:
    #     l_i = (env.controlled_vehicles[0].lane_index[0], env.controlled_vehicles[0].lane_index[1], env.controlled_vehicles[0].lane_index[2]-1)
    #     front_vehicle , rear_vehicle = env.road.neighbour_vehicles(env.controlled_vehicles[0], l_i)
        
    #     if rear_vehicle is not None:
    #         if env.controlled_vehicles[0].speed > rear_vehicle.speed
    #             overtake_gap = env.controlled_vehicles[0].position[0] - rear_vehicle.position[0]
    #             if overtake_gap < 25:
    #                 overtake += 1

    # if env.controlled_vehicles[0].lane_index[2] + 1 < env.config["lanes_count"]:
    #     l_i = (env.controlled_vehicles[0].lane_index[0], env.controlled_vehicles[0].lane_index[1], env.controlled_vehicles[0].lane_index[2]+1)
    #     _ , rear_vehicle = env.road.neighbour_vehicles(env.controlled_vehicles[0], l_i)
        
    #     if rear_vehicle is not None:
    #         if env.controlled_vehicles[0].speed > rear_vehicle.speed:
    #             overtake_gap =  env.controlled_vehicles[0].position[0] - rear_vehicle.position[0]
    #             if overtake_gap < 25:
    #                 overtake += 1
    # print(f"current overtakes: {overtake}")

# env.close()
    # print(f"overall overtakes: {overtake}")